***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节：[9.36 真实轻量样本包与项目练习材料](9_36_real_lightweight_sample_package_exercises.ipynb)
    * 下一节：[9.38 BIMA Measurement Set 校准复盘](9_38_bima_measurement_set_calibration_replay.ipynb)

***


## 9.37 真实 PyBDSF 产品复盘：WSRT Abell 2255

本节使用 PyBDSF 官方 GPLv3 仓库的真实数据测试夹具。输入 FITS 头将它标识为 2006 年 WSRT 的 Abell 2255 裁剪图像；上游参考产品包括频率平均后的 Stokes I 检测图、RMS 图、高斯模型、高斯残差和源级 FITS 目录。文件保持上游原始字节，源 commit、原路径、许可证和 SHA-256 都在样本包 manifest 中。

学习目标不是再次运行一个黑箱源搜索器，而是学会审计它的输入和产品契约：数据身份是否可核对，检测图如何从输入得到，模型与残差是否闭合，目录行数和形态代码是否与产品一致，以及哪些科学结论被裁剪过的测试头信息排除。


### 9.37.1 先审计 provenance，再看图

样本包位于 `sample_packages/pybdsf_abell2255_replay/`。`manifests/product_manifest.yaml` 记录 PyBDSF 源仓库、当前 commit 和测试夹具首次出现的历史 commit，`LICENSE-PyBDSF` 是上游仓库级 GPLv3 文本。未发现该 FITS 的单独许可声明或原始归档编号，这个 provenance 缺口也必须保留。`configs/pybdsf_parameters.yaml` 仅转录与本案例有关的 beam、RMS box、阈值和模型设置。上游 `parameters_used` 包含开发者机器的绝对路径，因此本仓库没有复制该文件。

输入是 $4\times10\times129\times129$ 的裁剪数组，包含 4 个 Stokes 和 10 个频率平面。它不是 Measurement Set，也不是完整归档包：原始可见度、flag、校准表和完整频率设置都不在其中。特别是裁剪输入头没有 `BUNIT`，物理 beam 和处理假设由 PyBDSF 参数外部提供。因此本节能复盘产品契约和目录 QA，但不能独立复核通量标度、重新校准或重新成像。


In [ ]:
import importlib.util
from pathlib import Path

package_dir = Path('sample_packages/pybdsf_abell2255_replay')
if not package_dir.exists():
    package_dir = Path('9_Practical') / package_dir
spec = importlib.util.spec_from_file_location('abell2255_replay', package_dir / 'analyze_products.py')
replay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(replay)
summary = replay.product_summary()
summary


### 9.37.2 产品闭合是最低限度的可复现性

脚本先核对 7 个文件的 SHA-256，然后检验两个独立契约。第一，上游检测图应等于输入 Stokes I 的 10 个平面平均：

$$I_{\rm det}(x,y) = {1\over10}\sum_{k=1}^{10} I_k(x,y).$$

第二，高斯模型和高斯残差应闭合到同一检测图：

$$I_{\rm det}(x,y) = I_{\rm model}(x,y) + I_{\rm residual}(x,y).$$

两个最大逐像素差都小于 $2\times10^{-8}$。这证明文件之间的数学身份一致，却不证明模型完备、源分解唯一或目录适合特定科学问题。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

image = fits.getdata(package_dir / 'products/mean_stokes_i.fits').squeeze()
rms = fits.getdata(package_dir / 'products/local_rms.fits').squeeze()
model = fits.getdata(package_dir / 'products/gaussian_model.fits').squeeze()
residual = fits.getdata(package_dir / 'products/gaussian_residual.fits').squeeze()

rms_level = np.median(rms)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
panels = (
    (image, 'Collapsed Stokes-I image', 'viridis', -3 * rms_level, np.percentile(image, 99.8)),
    (model, 'PyBDSF Gaussian model', 'viridis', 0.0, np.percentile(image, 99.8)),
    (residual, 'Gaussian residual', 'coolwarm', -5 * rms_level, 5 * rms_level),
    (rms, 'PyBDSF RMS map', 'magma', None, None),
)
for ax, (data, title, cmap, vmin, vmax) in zip(axes.flat, panels):
    view = ax.imshow(data, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('Pixel')
    ax.set_ylabel('Pixel')
    fig.colorbar(view, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


### 9.37.3 目录存在不等于目录可发布

参考目录有 43 个 sources，其中 `S`、`M`、`C` 形态代码分别有 31、4、8 个。这些代码来自拟合和分组结果，不是天体物理分类。峰值与积分流量的关系可用于寻找展源或不稳定拟合，island 内残差 RMS 与原 island RMS 的关系则更直接地检查拟合是否留下结构。


In [ ]:
catalog = fits.getdata(package_dir / 'products/source_catalog.fits', 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for code, marker in [('S', 'o'), ('M', 's'), ('C', '^')]:
    selected = catalog['S_Code'] == code
    axes[0].scatter(
        catalog['Peak_flux'][selected],
        catalog['Total_flux'][selected],
        marker=marker,
        alpha=0.8,
        label=code,
    )
limits = (5e-4, 3e-1)
axes[0].plot(limits, limits, color='0.3', linestyle='--')
axes[0].set(xscale='log', yscale='log', xlim=limits, ylim=limits)
axes[0].set_xlabel('Peak flux')
axes[0].set_ylabel('Integrated flux')
axes[0].legend(title='Source code')
axes[0].grid(alpha=0.3)

axes[1].scatter(catalog['Isl_rms'], catalog['Resid_Isl_rms'], alpha=0.8)
rms_limits = (1e-6, 1e-2)
axes[1].plot(rms_limits, rms_limits, color='0.3', linestyle='--')
axes[1].set(xscale='log', yscale='log', xlim=rms_limits, ylim=rms_limits)
axes[1].set_xlabel('Island RMS before fitting')
axes[1].set_ylabel('Island RMS after fitting')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Residual peak / RMS map: {summary['residual_peak_over_rms']:.1f}")


### 9.37.4 定量复盘任务（25 分）

1. 核对 manifest 与 FITS 头，列出两个直接来自观测的元数据和两个由处理参数外部提供的量。解释缺失 `BUNIT` 为什么阻止独立通量解释。（5 分）
2. 独立重算两个产品闭合差，说明“像素闭合”能和不能证明什么。（4 分）
3. 全局残差 robust RMS 与 RMS 图的中位数接近，但最大残差约为 59 倍 RMS。结合图像区分“全局噪声合理”和“局部模型失败”，提出两项后续诊断。（6 分）
4. 对峰值/积分流量图和 island RMS 图各选一个异常对象，说明需要回到 island、component 还是原图像层级审查。（4 分）
5. 写一段不超过 120 字的发布边界：必须同时说明该包支持的 QA、不支持的重处理，以及目录不能自动解释为天体物理分类。（6 分）

本案例比合成源表实验多了真实 FITS 头、复杂残差和上游目录，但仍没有跨过“图像产品”到“可见度重处理”的边界。这个区分必须保留：真实数据不会因为被裁剪成小文件，就自动支持所有科学结论。

***
